# Create CPRIT (Cancer Prevention and Research Institute of Texas) Awards

**CPRIT** — Cancer Prevention and Research Institute of Texas (funder_id `4320308129`,
ROR `0003xa228`, DOI `10.13039/100004917`, priority `377`). Source: the CPRIT funded-grants
database at cprit.texas.gov (Cloudflare-walled; headless Playwright + DataTables JS
extraction — see scripts/local/cprit_to_s3.py). **2,285 grants** totalling ~$4.14B: native
CPRIT ids (RP/PP/RR/DP/CP/CC + legacy numeric), **100% title / institution / USD amount /
full award date**, 99% person PI (17 pending recruitment awards have no named PI yet),
91% description (cancer types). Programs: Academic Research / CPRIT Scholar / Prevention /
Product Development Research. Existing OpenAlex coverage was Crossref-derived only.

Parquet: `s3://openalex-ingest/awards/cprit/cprit_grants.parquet`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cprit_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/cprit/cprit_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.cprit_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.cprit_raw LIMIT 5;

## Step 2: Create CPRIT Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cprit_awards
USING delta
AS
WITH
cprit_funder AS (
    -- CPRIT F4320308129 in common.funder (ROR 0003xa228, DOI 10.13039/100004917)
    SELECT CAST(funder_id AS BIGINT) as funder_id, display_name, ror_id, doi
    FROM openalex.common.funder WHERE funder_id = 4320308129
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        COALESCE(NULLIF(TRIM(g.title), ''), CONCAT('CPRIT grant ', g.funder_award_id)) as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DECIMAL(18,2)) > 0 THEN TRY_CAST(g.amount AS DECIMAL(18,2)) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DECIMAL(18,2)) > 0 THEN g.currency ELSE NULL END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.scheme as funder_scheme,
        'cprit' as provenance,
        TRY_TO_DATE(g.start_date_raw, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date_raw, 'yyyy-MM-dd') as end_date,
        YEAR(TRY_TO_DATE(g.start_date_raw, 'yyyy-MM-dd')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date_raw, 'yyyy-MM-dd')) as end_year,
        -- mixed: named PI -> person; pending recruitment award -> org (institution carried, names null)
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name, g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid, CAST(NULL AS DATE) as role_start,
                    struct(g.institution as name, 'United States' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids) as affiliation
                )
            WHEN g.institution IS NOT NULL THEN
                struct(
                    CAST(NULL AS STRING) as given_name, CAST(NULL AS STRING) as family_name,
                    CAST(NULL AS STRING) as orcid, CAST(NULL AS DATE) as role_start,
                    struct(g.institution as name, 'United States' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.cprit_raw g
    CROSS JOIN cprit_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw WHERE provenance = 'cprit' AND priority = 377;
INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id, amount, currency, funder, funding_type, funder_scheme, provenance, start_date, end_date, start_year, end_year, lead_investigator, co_lead_investigator, investigators, landing_page_url, doi, works_api_url, created_date, updated_date, 377 as priority
FROM openalex.awards.cprit_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(DISTINCT funder_award_id) uniq_award, COUNT(DISTINCT id) uniq_id, COUNT(DISTINCT funder_id) funders,
  SUM(CASE WHEN display_name IS NULL OR LENGTH(TRIM(display_name))=0 THEN 1 ELSE 0 END) blank,
  COUNT(amount) has_amount, COUNT(lead_investigator.given_name) has_person_pi, COUNT(lead_investigator.affiliation.name) has_inst,
  COUNT(start_date) has_start, ROUND(SUM(amount)/1000000,1) total_musd, MIN(start_year) min_yr, MAX(start_year) max_yr
FROM openalex.awards.cprit_awards;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) cnt, ROUND(SUM(amount)/1000000,1) musd
FROM openalex.awards.cprit_awards GROUP BY 1 ORDER BY 2 DESC LIMIT 12;

In [ ]:
%sql
SELECT COUNT(*) as in_raw FROM openalex.awards.openalex_awards_raw WHERE provenance='cprit' AND priority=377;